In [1]:
import atproto
from atproto import models

import config
from core.database import session_scope
from core.interactions.votes import (
    ThemeVoteGenerator,
    add_theme_vote
)

In [7]:
vote = ThemeVoteGenerator().generate()

print('built (unsaved) vote')
print(f'  type:         {vote.type.value}')
print(f'  voting:       {vote.vote_start_date} -> {vote.vote_end_date}')
print(f'  theme window: {vote.theme_start_date} -> {vote.theme_end_date}')
print('  options:', [o.theme_name for o in vote.options])

built (unsaved) vote
  type:         default
  voting:       2026-06-14 07:24:15.603245+00:00 -> 2026-06-15 07:24:15.603245+00:00
  theme window: 2026-06-15 07:24:15.603245+00:00 -> 2026-06-17 07:24:15.603245+00:00
  options: ['Onyx', 'Pikachu', 'Monochrome', 'Winter', 'Synthwave']


In [8]:
client = atproto.Client()
client.login(config.ATPROTO_CLIENT_USERNAME, config.ATPROTO_CLIENT_PASSWORD)

ProfileViewDetailed(did='did:plc:lk63sepwkbzg6cm67ws7ihq4', handle='randomcolormemes.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:lk63sepwkbzg6cm67ws7ihq4/bafkreiedgvyny4xjecczfjvk3o5etogdhff27fj6ncoewxutyp3weyszi4', banner='https://cdn.bsky.app/img/banner/plain/did:plc:lk63sepwkbzg6cm67ws7ihq4/bafkreifzzgm5qaus6x7ozsdzvtki2mnlnu7bpxhm3bqpxhhwiraf5dn6sa', created_at='2026-01-24T09:35:14.545Z', debug=None, description='Posts a random color every hour.', display_name='Random Color Memes', followers_count=22, follows_count=0, indexed_at='2026-06-08T05:47:07.865Z', joined_via_starter_pack=None, labels=[], pinned_post=Main(cid='bafyreihjnekfvxciwwr6jpjzhvmzogk3e43oxh7

In [10]:
# the poll itself
root = client.send_post(text='TEST POST' + vote.post_text)
vote.post_uri = root.uri
vote.post_cid = root.cid
print('posted poll:', vote.post_uri)

# every option comment is a direct reply to the poll (siblings under the post)
root_ref = models.create_strong_ref(root)
for option in vote.options:
    reply_ref = models.AppBskyFeedPost.ReplyRef(parent=root_ref, root=root_ref)
    comment = client.send_post(text='TEST COMMENT' + option.comment_text, reply_to=reply_ref)
    option.comment_uri = comment.uri
    option.comment_cid = comment.cid
    print(f'  commented {option.theme_name}: {comment.uri}')



posted poll: at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagvcbgs2n
  commented Onyx: at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagvnrx72y
  commented Pikachu: at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagvu5vr2b
  commented Monochrome: at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagw7rrk2n
  commented Winter: at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagwglm52c
  commented Synthwave: at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagws34g2h


In [11]:
with session_scope() as session:
    vote = add_theme_vote(session, vote)

print(f'saved vote {vote.id}')
for option in vote.options:
    print(f'    - {option.theme_name} ({option.theme_tag})  comment={option.comment_uri}')

saved vote 1
    - Onyx (onyx)  comment=at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagvnrx72y
    - Pikachu (pikachu)  comment=at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagvu5vr2b
    - Monochrome (monochrome)  comment=at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagw7rrk2n
    - Winter (winter)  comment=at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagwglm52c
    - Synthwave (synthwave)  comment=at://did:plc:lk63sepwkbzg6cm67ws7ihq4/app.bsky.feed.post/3moaagws34g2h
